# Gridalytics - Feature Analysis

Analyze the 93 engineered features: correlations, importance, distributions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from datetime import date

from src.data.db.session import get_session
from src.features.pipeline import FeaturePipeline

%matplotlib inline

## 1. Build Feature Matrix

In [ ]:
with get_session() as session:
    pipeline = FeaturePipeline('hourly', session)
    df = pipeline.build(date(2021, 1, 1), date(2026, 3, 28))

target = 'delhi_mw'
features = pipeline.get_feature_names(df)
features = [f for f in features if f not in ['brpl_mw', 'bypl_mw', 'ndpl_mw', 'ndmc_mw', 'mes_mw']]
print(f'Shape: {df.shape}')
print(f'Features: {len(features)}')
print(f'Target: {target}')

## 2. Top Correlations with Demand

In [ ]:
corrs = df[features].corrwith(df[target]).abs().sort_values(ascending=True).tail(20)
fig = px.bar(x=corrs.values, y=corrs.index, orientation='h',
             title='Top 20 Features by Correlation with Demand',
             labels={'x': 'Absolute Correlation', 'y': 'Feature'})
fig.update_layout(template='plotly_dark', height=600)
fig.show()

## 3. Feature Distributions

In [ ]:
key_features = ['delhi_mw_lag_1', 'delhi_mw_lag_24', 'CDD', 'heat_index', 'temperature_2m', 'hour_sin']
for feat in key_features:
    if feat in df.columns:
        fig = px.histogram(df, x=feat, nbins=50, title=f'Distribution: {feat}',
                          marginal='box')
        fig.update_layout(template='plotly_dark', height=300)
        fig.show()

## 4. Correlation Matrix (Top Features)

In [ ]:
top_feats = corrs.tail(15).index.tolist() + [target]
corr_matrix = df[top_feats].corr()
fig = px.imshow(corr_matrix, text_auto='.2f', title='Correlation Matrix (Top 15 Features)',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.update_layout(template='plotly_dark', height=700)
fig.show()

## 5. Feature Groups Analysis

In [ ]:
groups = {}
for f in features:
    if '_lag_' in f: groups.setdefault('Lag', []).append(f)
    elif '_diff_' in f: groups.setdefault('Diff', []).append(f)
    elif 'rmean' in f or 'rstd' in f or 'rmin' in f or 'rmax' in f: groups.setdefault('Rolling', []).append(f)
    elif 'fourier' in f: groups.setdefault('Fourier', []).append(f)
    elif '_sin' in f or '_cos' in f: groups.setdefault('Cyclical', []).append(f)
    elif f in ['temperature_2m','relative_humidity_2m','dew_point_2m','precipitation_mm','cloud_cover_pct','wind_speed_10m','shortwave_radiation']: groups.setdefault('Raw Weather', []).append(f)
    elif f in ['CDD','HDD','heat_index','temp_squared','temp_x_humidity','temp_x_hour']: groups.setdefault('Weather Derived', []).append(f)
    else: groups.setdefault('Other', []).append(f)

for name, feats in sorted(groups.items()):
    avg_corr = df[feats].corrwith(df[target]).abs().mean()
    print(f'{name:20s}: {len(feats):3d} features, avg |correlation| = {avg_corr:.4f}')